# RAG Notebook 2 — XLM-RoBERTa (Extractive QA)

**Model:** `deepset/xlm-roberta-base-squad2` — fine-tuned on SQuAD2 for multilingual span extraction

**Pipeline:** Question → FAISS retrieval (top-3 chunks) → XLM-RoBERTa extracts span → Evaluate

**Metrics:** BLEU · BERTScore F1 · Answer Similarity · Context Relevance · Faithfulness

## Files needed
```
/content/rag_eval_set.json
/content/drive/MyDrive/AgriQADataset/AgriQADataset/Central Govt Schemes/
/content/drive/MyDrive/AgriQADataset/AgriQADataset/Karnataka State Schemes/
```
**Runtime:** T4 GPU

## ── CELL 1 ── Install packages

In [1]:
!pip install -q transformers==4.46.0 sentencepiece accelerate
!pip install -q faiss-cpu sentence-transformers
!pip install -q sacrebleu bert-score
print('✅ Packages installed')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 596.5 kB/s eta 0:00:00
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.6 MB/s eta 0:00:00
✅ Packages installed


## ── CELL 2 ── Mount Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted')

Mounted at /content/drive
✅ Drive mounted


## ── CELL 3 ── Paths and config

In [4]:
import os, json, re, glob
import numpy as np
import torch

EVAL_JSON     = '/content/rag_eval_set.json'
CENTRAL_DIR   = '/content/drive/MyDrive/AgriQADataset/AgriQADataset/Central Govt Schemes'
KARNATAKA_DIR = '/content/drive/MyDrive/AgriQADataset/AgriQADataset/Karnataka State Schemes'
RESULTS_DIR   = '/content/drive/MyDrive/AgriQA/rag_comparison_results'

RETRIEVER_MODEL = 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
CHUNK_SIZE      = 400
CHUNK_OVERLAP   = 80
TOP_K           = 3

os.makedirs(RESULTS_DIR, exist_ok=True)
assert os.path.exists(EVAL_JSON),     f'Not found: {EVAL_JSON}'
assert os.path.exists(CENTRAL_DIR),   f'Not found: {CENTRAL_DIR}'
assert os.path.exists(KARNATAKA_DIR), f'Not found: {KARNATAKA_DIR}'
print('✅ Paths verified')

✅ Paths verified


## ── CELL 4 ── GPU check

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')
if device == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

Device : cpu


## ── CELL 5 ── Load and chunk scheme files

In [6]:
def read_scheme(path):
    with open(path, encoding='utf-8') as f:
        raw = f.read().replace('\r\n','\n').replace('\r','\n')
    txt = re.sub(r'"([^"]{2,40})":', r'\1:', raw)
    txt = re.sub(r'["\[\]{}]', ' ', txt)
    txt = re.sub(r',\s*\n', '\n', txt)
    txt = re.sub(r'[ \t]+', ' ', txt)
    txt = re.sub(r'\n{3,}', '\n\n', txt)
    return txt.strip()

def chunk_text(text, scheme_no):
    chunks, start = [], 0
    while start < len(text):
        end = min(start + CHUNK_SIZE, len(text))
        if end < len(text):
            nb = text.rfind('\n', start, end)
            if nb > start + CHUNK_SIZE // 2:
                end = nb
        chunk = text[start:end].strip()
        if len(chunk) > 30:
            chunks.append({'scheme_no': scheme_no,
                           'chunk_id': f'{scheme_no}_{len(chunks)}',
                           'text': chunk})
        start = end - CHUNK_OVERLAP if end < len(text) else len(text)
    return chunks

file_map = {}
for d in [CENTRAL_DIR, KARNATAKA_DIR]:
    for p in glob.glob(os.path.join(d, '*.txt')):
        b = os.path.basename(p).replace('.txt','')
        if b.isdigit():
            file_map[int(b)] = p

all_chunks, full_ctx_map = [], {}
for sno in sorted(file_map):
    txt = read_scheme(file_map[sno])
    full_ctx_map[sno] = txt
    all_chunks.extend(chunk_text(txt, sno))

print(f'✅ {len(file_map)} scheme files → {len(all_chunks)} chunks')

✅ 80 scheme files → 720 chunks


## ── CELL 6 ── Build FAISS retrieval index

In [7]:
import faiss
from sentence_transformers import SentenceTransformer

print('Loading retriever …')
retriever = SentenceTransformer(RETRIEVER_MODEL)

print('Encoding chunks …')
embeddings = retriever.encode(
    [c['text'] for c in all_chunks],
    batch_size=64, show_progress_bar=True,
    normalize_embeddings=True, convert_to_numpy=True,
)
dim   = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings.astype(np.float32))
print(f'✅ FAISS index — {index.ntotal} vectors, dim={dim}')

def retrieve(question, top_k=TOP_K):
    q = retriever.encode([question], normalize_embeddings=True,
                         convert_to_numpy=True).astype(np.float32)
    scores, idxs = index.search(q, top_k)
    hits = []
    for sc, idx in zip(scores[0], idxs[0]):
        c = dict(all_chunks[idx])
        c['score'] = float(sc)
        hits.append(c)
    return hits

test = retrieve('ಬೆಳೆ ವಿಮೆ ಯೋಜನೆ')
print(f'Test: scheme={test[0]["scheme_no"]} score={test[0]["score"]:.3f}')

Loading retriever …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding chunks …


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

✅ FAISS index — 720 vectors, dim=384
Test: scheme=78 score=0.720


## ── CELL 7 ── Load evaluation set

In [8]:
with open(EVAL_JSON, encoding='utf-8') as f:
    eval_set = json.load(f)
print(f'✅ {len(eval_set)} evaluation questions')
for q in eval_set:
    print(f'  {q["qid"]} | scheme={q["scheme_no"]} | {q["question"]}')

✅ 15 evaluation questions
  E01 | scheme=1 | ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು?
  E02 | scheme=1 | PMFBY ಯೋಜನೆಗೆ ಅರ್ಜಿ ಸಲ್ಲಿಸಲು ಯಾವ ದಾಖಲೆಗಳು ಬೇಕು?
  E03 | scheme=1 | ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಸಹಾಯವಾಣಿ ಸಂಖ್ಯೆ ಏನು?
  E04 | scheme=2 | ಪ್ರಧಾನಮಂತ್ರಿ ಕೃಷಿ ಸಿಂಚಾಯಿ ಯೋಜನೆ ಯಾವ ವರ್ಷ ಆರಂಭವಾಯಿತು?
  E05 | scheme=2 | PMKSY ಯೋಜನೆಯ ಮುಖ್ಯ ಉದ್ದೇಶ ಏನು?
  E06 | scheme=3 | PM-KISAN ಯೋಜನೆಯಡಿ ರೈತರಿಗೆ ಎಷ್ಟು ಹಣ ಸಿಗುತ್ತದೆ?
  E07 | scheme=3 | ಪ್ರಧಾನಮಂತ್ರಿ ಕಿಸಾನ್ ಸಮ್ಮಾನ್ ನಿಧಿ ಯೋಜನೆಗೆ ಯಾರು ಅರ್ಹರು?
  E08 | scheme=41 | ರೈತ ಸಿರಿ ಯೋಜನೆ ಯಾವ ರಾಜ್ಯದಲ್ಲಿ ಜಾರಿಯಲ್ಲಿದೆ?
  E09 | scheme=41 | ರೈತ ಸಿರಿ ಯೋಜನೆಯ ಉದ್ದೇಶ ಏನು?
  E10 | scheme=50 | ಕರ್ನಾಟಕ ರೈತ ಸುರಕ್ಷಾ ವಿಮೆ ಯೋಜನೆ ಏನು ಒದಗಿಸುತ್ತದೆ?
  E11 | scheme=1 | PMFBY ಯೋಜನೆಗೆ ಅರ್ಜಿ ಎಲ್ಲಿ ಸಲ್ಲಿಸಬಹುದು?
  E12 | scheme=2 | ಪ್ರಧಾನಮಂತ್ರಿ ಕೃಷಿ ಸಿಂಚಾಯಿ ಯೋಜನೆ ಯಾವ ನೀರಾವರಿ ವಿಧಾನಗಳನ್ನು ಉತ್ತೇಜಿಸುತ್ತದೆ?
  E13 | scheme=3 | PM-KISAN ಯೋಜನೆ ಹಣ ಹೇಗೆ ನೀಡಲಾಗುತ್ತದೆ?
  E14 | scheme=41 | ರೈತ ಸಿರಿ ಯೋಜನೆಯಡಿ ರೈತರಿಗೆ ಯಾವ ಸಹಾಯ ದೊರಕುತ್ತದೆ?
  E15 | scheme=50 | ಕರ್ನಾಟಕ ರೈತ ಸುರಕ್ಷಾ ವಿಮೆ ಯೋಜನೆಗೆ ಯಾರು ಅರ್ಹರು?


## ── CELL 8 ── Define all 5 metrics

In [9]:
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn

_bleu = BLEU(effective_order=True)

def compute_all_metrics(predictions, references, questions, contexts):
    """
    5 metrics:
    1. BLEU              — n-gram overlap (sacrebleu)
    2. BERTScore F1      — contextual semantic similarity (lang=kn)
    3. Answer Similarity — cosine(pred_emb, gold_emb)  [RAGAS-style]
    4. Context Relevance — cosine(question_emb, context_emb)  [RAGAS-style]
    5. Faithfulness      — cosine(pred_emb, context_emb)  [RAGAS-style]
    """
    bleu_scores = [_bleu.sentence_score(p, [r]).score
                   for p, r in zip(predictions, references)]
    _, _, F1 = bert_score_fn(predictions, references, lang='kn', verbose=False)
    bert_scores = F1.numpy().tolist()

    pred_emb = retriever.encode(predictions, normalize_embeddings=True, convert_to_numpy=True)
    ref_emb  = retriever.encode(references,  normalize_embeddings=True, convert_to_numpy=True)
    q_emb    = retriever.encode(questions,   normalize_embeddings=True, convert_to_numpy=True)
    ctx_emb  = retriever.encode(contexts,    normalize_embeddings=True, convert_to_numpy=True)

    ans_sim  = (pred_emb * ref_emb).sum(axis=1).tolist()
    ctx_rel  = (q_emb   * ctx_emb).sum(axis=1).tolist()
    faithful = (pred_emb * ctx_emb).sum(axis=1).tolist()

    def avg(lst): return round(float(np.mean(lst)), 4)
    def rnd(lst): return [round(float(x), 4) for x in lst]

    return {
        'bleu':              avg(bleu_scores),
        'bertscore_f1':      avg(bert_scores),
        'answer_similarity': avg(ans_sim),
        'context_relevance': avg(ctx_rel),
        'faithfulness':      avg(faithful),
        'bleu_per_q':        rnd(bleu_scores),
        'bert_per_q':        rnd(bert_scores),
        'ans_sim_per_q':     rnd(ans_sim),
        'ctx_rel_per_q':     rnd(ctx_rel),
        'faithful_per_q':    rnd(faithful),
    }

print('✅ 5 metrics defined')

✅ 5 metrics defined


## ── CELL 9 ── Load XLM-RoBERTa

In [10]:
from transformers import pipeline

print('Loading XLM-RoBERTa …')
qa_pipe = pipeline(
    'question-answering',
    model='deepset/xlm-roberta-base-squad2',
    device=0 if device == 'cuda' else -1,
)
print('✅ XLM-RoBERTa loaded (deepset/xlm-roberta-base-squad2)')

Loading XLM-RoBERTa …


config.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/79.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✅ XLM-RoBERTa loaded (deepset/xlm-roberta-base-squad2)


## ── CELL 10 ── Run XLM-RoBERTa RAG on all 15 questions

In [11]:
predictions, references, questions_list, contexts_list = [], [], [], []
rows = []

for qa in eval_set:
    hits    = retrieve(qa['question'])
    context = ' '.join(h['text'] for h in hits)
    top_ctx = hits[0]['text']

    try:
        out    = qa_pipe(question=qa['question'], context=context, max_answer_len=200)
        answer = out['answer'].strip() if out['score'] > 0.001 else 'ಈ ಮಾಹಿತಿಯು ಲಭ್ಯವಿಲ್ಲ'
    except Exception as e:
        answer = ''
        print(f'Error on {qa["qid"]}: {e}')

    predictions.append(answer)
    references.append(qa['answer'])
    questions_list.append(qa['question'])
    contexts_list.append(top_ctx)

    rows.append({
        'qid':               qa['qid'],
        'scheme_no':         qa['scheme_no'],
        'question':          qa['question'],
        'gold':              qa['answer'],
        'predicted':         answer,
        'top_scheme_hit':    hits[0]['scheme_no'],
        'retrieval_correct': qa['scheme_no'] == hits[0]['scheme_no'],
        'retrieval_score':   round(hits[0]['score'], 4),
    })
    print(f'{qa["qid"]} ✓')

print(f'\n✅ Inference done — {len(predictions)} answers')

E01 ✓
E02 ✓
E03 ✓
E04 ✓
E05 ✓
E06 ✓
E07 ✓
E08 ✓
E09 ✓
E10 ✓
E11 ✓
E12 ✓
E13 ✓
E14 ✓
E15 ✓

✅ Inference done — 15 answers


## ── CELL 11 ── Compute all 5 metrics

In [12]:
print('Computing metrics …')
metrics = compute_all_metrics(predictions, references, questions_list, contexts_list)

print()
print('=' * 55)
print('  XLM-RoBERTa RAG — RESULTS')
print('=' * 55)
print(f'  BLEU                        : {metrics["bleu"]}')
print(f'  BERTScore F1                : {metrics["bertscore_f1"]}')
print(f'  Answer Similarity  (RAGAS)  : {metrics["answer_similarity"]}')
print(f'  Context Relevance  (RAGAS)  : {metrics["context_relevance"]}')
print(f'  Faithfulness       (RAGAS)  : {metrics["faithfulness"]}')
print('=' * 55)

Computing metrics …


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]


  XLM-RoBERTa RAG — RESULTS
  BLEU                        : 4.8908
  BERTScore F1                : 0.6826
  Answer Similarity  (RAGAS)  : 0.6285
  Context Relevance  (RAGAS)  : 0.83
  Faithfulness       (RAGAS)  : 0.604


## ── CELL 12 ── Per-question breakdown

In [13]:
import pandas as pd

df = pd.DataFrame(rows)
df['bleu']     = metrics['bleu_per_q']
df['bert']     = metrics['bert_per_q']
df['ans_sim']  = metrics['ans_sim_per_q']
df['ctx_rel']  = metrics['ctx_rel_per_q']
df['faithful'] = metrics['faithful_per_q']

print('Per-question scores:')
print(df[['qid','bleu','bert','ans_sim','ctx_rel','faithful','retrieval_correct']].to_string(index=False))
print(f'\nRetrieval accuracy: {df["retrieval_correct"].mean()*100:.1f}%')

Per-question scores:
qid    bleu   bert  ans_sim  ctx_rel  faithful  retrieval_correct
E01  1.8075 0.6916   0.9026   0.7639    0.9162              False
E02  0.0000 0.6697   0.2561   0.8183    0.2402              False
E03  0.0000 0.6727   0.7051   0.9060    0.6163              False
E04  7.2532 0.6530   0.4845   0.8895    0.6377              False
E05 51.3417 0.9259   0.8378   0.6550    0.6555               True
E06  0.0000 0.6364   0.3858   0.7286    0.4166              False
E07  0.5040 0.6205   0.5746   0.9160    0.6377              False
E08  0.0000 0.6348   0.6332   0.9089    0.6315              False
E09  2.7179 0.7302   0.8703   0.8276    0.7529              False
E10  0.0000 0.6023   0.6199   0.9294    0.6652              False
E11  0.6738 0.7724   0.6001   0.6795    0.7057              False
E12  0.0000 0.6374   0.3212   0.9408    0.3627              False
E13  0.0000 0.6568   0.7338   0.6182    0.2226              False
E14  1.8110 0.7002   0.8215   0.9322    0.9336         

## ── CELL 13 ── Show sample predictions

In [14]:
print('Sample predictions (first 5):')
print('=' * 70)
for _, row in df.head(5).iterrows():
    print(f'QID  : {row["qid"]}')
    print(f'Q    : {row["question"]}')
    print(f'Gold : {row["gold"][:100]}')
    print(f'Pred : {row["predicted"][:100]}')
    print(f'BLEU={row["bleu"]:.3f}  BERT={row["bert"]:.3f}  AnsS={row["ans_sim"]:.3f}')
    print('-' * 70)

Sample predictions (first 5):
QID  : E01
Q    : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಉದ್ದೇಶ ಏನು?
Gold : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆವು ಬೆಳೆ ಹಾನಿಯಾದ ಸಂದರ್ಭಗಳಲ್ಲಿ ರೈತರಿಗೆ ವಿಮೆಯ ಮೂಲಕ ಆರ್ಥಿಕ ರಕ್ಷಣೆ ಒದಗಿಸಲು ರೂ
Pred : ಬೆಳೆಗಳನ್ನು ಹವಾಮಾನದಿಂದ ರಕ್ಷಿಸುವುದು
ಕೃಷಿ ಉತ್ಪಾದನೆ ಹೆಚ್ಚಿಸುವುದು
ಉತ್ತಮ ಗುಣಮಟ್ಟದ ಉತ್ಪನ್ನ ಪಡೆಯುವುದು
ರೈತರ ಆ
BLEU=1.808  BERT=0.692  AnsS=0.903
----------------------------------------------------------------------
QID  : E02
Q    : PMFBY ಯೋಜನೆಗೆ ಅರ್ಜಿ ಸಲ್ಲಿಸಲು ಯಾವ ದಾಖಲೆಗಳು ಬೇಕು?
Gold : ಆಧಾರ್ ಕಾರ್ಡ್, ಬ್ಯಾಂಕ್ ಖಾತೆ ವಿವರ, ಭೂ ದಾಖಲೆ (ಪಹಣಿ), ಬೆಳೆ ವಿವರ, FRUITS ನೋಂದಣಿ ಮಾಹಿತಿ (ಅಗತ್ಯವಿದ್ದರೆ), ಕಂ
Pred : ತೋಟಗಾರಿಕೆ
BLEU=0.000  BERT=0.670  AnsS=0.256
----------------------------------------------------------------------
QID  : E03
Q    : ಪ್ರಧಾನಮಂತ್ರಿ ಫಸಲ್ ಬಿಮಾ ಯೋಜನೆ ಸಹಾಯವಾಣಿ ಸಂಖ್ಯೆ ಏನು?
Gold : 14447 ಮತ್ತು 1800-180-1551
Pred : ೧೮೦೦-೧೮೦-೧೫೫೧
BLEU=0.000  BERT=0.673  AnsS=0.705
----------------------------------------------------------------------
QID  : E04
Q    : ಪ್ರಧಾನಮಂತ್ರಿ ಕೃಷಿ ಸಿಂಚಾಯಿ ಯೋಜನೆ ಯಾವ ವರ್ಷ ಆರಂಭವ

## ── CELL 14 ── Save results

In [15]:
csv_path = f'{RESULTS_DIR}/xlmr_rag_results.csv'
df.to_csv(csv_path, index=False, encoding='utf-8')
print(f'✅ CSV saved → {csv_path}')

result_dict = {
    'model': 'deepset/xlm-roberta-base-squad2',
    'type': 'extractive_qa',
    'eval_size': len(eval_set),
    'bleu':              metrics['bleu'],
    'bertscore_f1':      metrics['bertscore_f1'],
    'answer_similarity': metrics['answer_similarity'],
    'context_relevance': metrics['context_relevance'],
    'faithfulness':      metrics['faithfulness'],
}
json_path = f'{RESULTS_DIR}/xlmr_rag_metrics.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(result_dict, f, indent=2, ensure_ascii=False)
print(f'✅ Metrics JSON → {json_path}')
print()
print('Use these numbers in Notebook 4 (comparison).')

✅ CSV saved → /content/drive/MyDrive/AgriQA/rag_comparison_results/xlmr_rag_results.csv
✅ Metrics JSON → /content/drive/MyDrive/AgriQA/rag_comparison_results/xlmr_rag_metrics.json

Use these numbers in Notebook 4 (comparison).
